# ДЗ №7 — Улучшение модели SafeCall Voice Guard (исправленная версия)

Этот ноутбук сделан как исправление к замечаниям по ДЗ7. Главная цель — не
перерисовать старый отчёт, а показать проверяемые артефакты:

- ячейки выполнены и содержат outputs;
- XLSR-53 описан честно: **frozen feature extractor + обученная MLP-head**;
- `SimplifiedAASIST` больше не называется AASIST, это `MelCNNBaseline`;
- метрики пересчитываются из сохранённых `y_*.npy` и `eval_probs_*.npy`;
- preprocessing/chunking исправлен так, чтобы не терять хвост аудио;
- синтетические иллюстрации не используются как доказательство real-data EDA.

Критерии ДЗ закрываются так:

| Критерий | Где закрывается |
|---|---|
| Пайплайн предобработки / FE / генерации данных | секции 1–3 |
| Улучшенная архитектура модели | секция 4 |
| Постобработка предсказаний | секция 5 |
| Анализ качества в разрезе метрик | секции 6–7 |


In [1]:
from pathlib import Path
import csv, json, struct, ast, math
from collections import Counter, defaultdict

# Notebook can be run from repo root or from 7-fix.
ROOT = Path.cwd()
if not (ROOT / "safecall_train").exists():
    ROOT = ROOT.parent

TRAIN_DIR = ROOT / "safecall_train"
DATA_DIR = TRAIN_DIR / "data"
EMB_DIR = DATA_DIR / "embeddings"
MODEL_DIR = TRAIN_DIR / "models"

print("ROOT:", ROOT)
print("TRAIN_DIR exists:", TRAIN_DIR.exists())
print("metadata exists:", (DATA_DIR / "metadata.csv").exists())
print("embeddings dir exists:", EMB_DIR.exists())
print("models dir exists:", MODEL_DIR.exists())

ROOT: C:\Users\root\Desktop\dz itmo
TRAIN_DIR exists: True
metadata exists: True
embeddings dir exists: True
models dir exists: True


## 1. Проверка артефактов и факта обучения

В старом ноутбуке было непонятно, запускалось ли что-то реально. Здесь сначала
проверяем физические артефакты, которые остаются после обучения:

- `X_train/dev/eval.npy` и `y_train/dev/eval.npy` — XLSR-53 embeddings и labels;
- `best_xlsr_head.pth` — обученная MLP-head;
- `xlsr_head_results.json` — метрики head при `threshold=0.50`;
- `threshold_tuning_results.json` — улучшение после подбора порога.

Важно: это **не full fine-tune XLSR-53 backbone**. Backbone был заморожен, обучалась
только классификационная голова поверх embeddings.


In [2]:
def npy_header(path):
    with open(path, "rb") as f:
        magic = f.read(6)
        if magic != b"\x93NUMPY":
            raise ValueError(f"Not a npy file: {path}")
        version = tuple(f.read(2))
        if version == (1, 0):
            header_len = struct.unpack("<H", f.read(2))[0]
        else:
            header_len = struct.unpack("<I", f.read(4))[0]
        header = ast.literal_eval(f.read(header_len).decode("latin1"))
    return header["shape"], header["descr"]

for rel in [
    "data/embeddings/X_train.npy", "data/embeddings/y_train.npy",
    "data/embeddings/X_dev.npy", "data/embeddings/y_dev.npy",
    "data/embeddings/X_eval.npy", "data/embeddings/y_eval.npy",
    "models/eval_probs_xlsr.npy", "models/dev_probs_xlsr.npy",
    "models/best_xlsr_head.pth", "models/xlsr_head_results.json",
    "models/threshold_tuning_results.json",
]:
    p = TRAIN_DIR / rel
    if p.suffix == ".npy":
        shape, dtype = npy_header(p)
        extra = f"shape={shape}, dtype={dtype}"
    else:
        extra = f"size={p.stat().st_size:,} bytes" if p.exists() else "MISSING"
    print(f"{rel:45s} {extra}")

data/embeddings/X_train.npy                   shape=(6363, 1024), dtype=<f4
data/embeddings/y_train.npy                   shape=(6363,), dtype=<i4
data/embeddings/X_dev.npy                     shape=(4275, 1024), dtype=<f4
data/embeddings/y_dev.npy                     shape=(4275,), dtype=<i4
data/embeddings/X_eval.npy                    shape=(11235, 1024), dtype=<f4
data/embeddings/y_eval.npy                    shape=(11235,), dtype=<i4
models/eval_probs_xlsr.npy                    shape=(11235,), dtype=<f4
models/dev_probs_xlsr.npy                     shape=(4275,), dtype=<f4
models/best_xlsr_head.pth                     size=1,184,526 bytes
models/xlsr_head_results.json                 size=449 bytes
models/threshold_tuning_results.json          size=937 bytes


## 2. Dataset composition и честный split

Замечание преподавателя про RU-рынок справедливое: нельзя просто заявлять
RU-first, если весь RU-сплит сгенерирован или если split не доказан.

Что делаем в этой версии:

- показываем фактический состав `metadata.csv`;
- отдельно считаем `language`, `source`, `label` по `train/dev/eval`;
- честно фиксируем ограничение: в текущем экспортированном metadata нет колонки
  `gender`, поэтому стратификацию по полу нельзя заявлять как выполненную;
- для RU fake проверяем доступное поле `source/tts_engine`.


In [3]:
def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

metadata = read_csv(DATA_DIR / "metadata.csv")
print("metadata rows:", len(metadata))

def count_by(rows, key):
    return Counter((r.get(key) or "<empty>") for r in rows)

def print_counter(title, counter, limit=None):
    print(title)
    items = counter.most_common(limit)
    for k, v in items:
        print(f"  {k:22s} {v:6d}")

print_counter("split:", count_by(metadata, "split"))
print_counter("label:", count_by(metadata, "label"))
print_counter("language:", count_by(metadata, "language"))
print_counter("source:", count_by(metadata, "source"))

print("\nPer split summary:")
for split in ["train", "dev", "eval"]:
    rows = [r for r in metadata if r["split"] == split]
    print(f"\n[{split}] n={len(rows)}")
    print_counter("  label", count_by(rows, "label"))
    print_counter("  language", count_by(rows, "language"))
    print_counter("  source", count_by(rows, "source"), limit=8)

columns = set(metadata[0].keys())
print("\nHas gender column:", "gender" in columns)
print("Has tts_engine column:", "tts_engine" in columns)

metadata rows: 21873
split:
  eval                    11235
  train                    6363
  dev                      4275
label:
  spoof                   16946
  bonafide                 4927
language:
  en                      18220
  ru                       3653
source:
  asvspoof2019la          18220
  golos                    2994
  ru_fake_unknown           499
  ru_fake_edge_tts          100
  common_voice_ru            60

Per split summary:

[train] n=6363
  label
  spoof                    3839
  bonafide                 2524
  language
  en                       3807
  ru                       2556
  source
  asvspoof2019la           3807
  golos                    2099
  ru_fake_unknown           352
  ru_fake_edge_tts           67
  common_voice_ru            38

[dev] n=4275
  label
  spoof                    3438
  bonafide                  837
  language
  en                       3727
  ru                        548
  source
  asvspoof2019la           3727
  golos  

In [4]:
print("RU-only composition by split:")
for split in ["train", "dev", "eval"]:
    rows = [r for r in metadata if r["split"] == split and r["language"] == "ru"]
    print(f"\n[{split}] RU n={len(rows)}")
    print_counter("  label", count_by(rows, "label"))
    print_counter("  source", count_by(rows, "source"))
    tts = Counter((r.get("tts_engine") or r.get("source") or "<empty>") for r in rows if r["label"] == "spoof")
    print_counter("  fake engine/source", tts)

print("\nВывод:")
print("- RU данные не являются только synthetic: есть Golos/Common Voice real и ru_fake.")
print("- Стратификацию по gender нельзя заявлять без поля gender.")
print("- Для RU fake можно контролировать source/tts_engine; это надо явно показывать в baseline.")

RU-only composition by split:

[train] RU n=2556
  label
  bonafide                 2137
  spoof                     419
  source
  golos                    2099
  ru_fake_unknown           352
  ru_fake_edge_tts           67
  common_voice_ru            38
  fake engine/source
  unknown                   352
  edge_tts                   67

[dev] RU n=548
  label
  bonafide                  458
  spoof                      90
  source
  golos                     451
  ru_fake_unknown            74
  ru_fake_edge_tts           16
  common_voice_ru             7
  fake engine/source
  unknown                    74
  edge_tts                   16

[eval] RU n=549
  label
  bonafide                  459
  spoof                      90
  source
  golos                     444
  ru_fake_unknown            73
  ru_fake_edge_tts           17
  common_voice_ru            15
  fake engine/source
  unknown                    73
  edge_tts                   17

Вывод:
- RU данные не являются толь

## 3. Исправленный preprocessing/chunking

Старый `preprocess_audio(max_duration=5)` действительно терял хвост: 12 секунд
превращались в два чанка по 5 секунд, последние 2 секунды исчезали.

Ниже — исправленная логика:

- явный `chunk_duration=4.0`, а не неявный `max_duration`;
- последний хвост не выбрасывается;
- хвост можно дополнить padding до полного чанка;
- 12 секунд при `chunk_duration=4` дают 3 чанка без потери аудио.


In [5]:
def make_chunks(total_samples, sample_rate=16000, chunk_duration=4.0, overlap=0.0, pad_tail=True):
    chunk = int(chunk_duration * sample_rate)
    step = int((chunk_duration - overlap) * sample_rate)
    if chunk <= 0 or step <= 0:
        raise ValueError("chunk_duration must be > overlap >= 0")
    chunks = []
    start = 0
    while start < total_samples:
        end = start + chunk
        real_end = min(end, total_samples)
        real_len = real_end - start
        if real_len <= 0:
            break
        if real_len < chunk and not pad_tail:
            chunks.append((start, real_end, real_len, "tail_partial"))
        else:
            chunks.append((start, end, real_len, "full" if real_len == chunk else "tail_padded"))
        if end >= total_samples:
            break
        start += step
    return chunks

demo = make_chunks(total_samples=12 * 16000, sample_rate=16000, chunk_duration=4.0, overlap=0.0)
for i, (s, e, real_len, kind) in enumerate(demo, 1):
    print(f"chunk {i}: {s/16000:.1f}s -> {e/16000:.1f}s, real_audio={real_len/16000:.1f}s, {kind}")
print("Total real seconds covered:", sum(c[2] for c in demo) / 16000)

chunk 1: 0.0s -> 4.0s, real_audio=4.0s, full
chunk 2: 4.0s -> 8.0s, real_audio=4.0s, full
chunk 3: 8.0s -> 12.0s, real_audio=4.0s, full
Total real seconds covered: 12.0


## 4. Улучшенная архитектура модели

Корректная формулировка:

> Используется `facebook/wav2vec2-large-xlsr-53` как замороженный feature extractor.
> Для каждого аудио сохранён embedding размерности 1024. Поверх embeddings обучена
> MLP-head `1024 -> 256 -> 128 -> 1`.

Это лучше подавать именно так, а не как full fine-tune XLSR-53. Для ДЗ это всё
равно является улучшенной архитектурой относительно простого baseline, потому что
используется сильный self-supervised speech encoder + обучаемая доменная голова.

`SimplifiedAASIST` переименован концептуально в `MelCNNBaseline`: это не AASIST,
а маленький CNN по mel-спектрограмме. Его нельзя использовать как доказательство
качества AASIST.


In [6]:
with open(MODEL_DIR / "xlsr_head_results.json", encoding="utf-8") as f:
    xlsr_results = json.load(f)

print("Saved XLSR-head training result:")
for k, v in xlsr_results.items():
    if k != "hyperparams":
        print(f"  {k}: {v}")
print("  hyperparams:", xlsr_results["hyperparams"])

print("\nInterpretation:")
print("- best_xlsr_head.pth is the trained MLP classification head.")
print("- XLSR-53 backbone was frozen; embeddings are stored in data/embeddings.")
print("- This is real training of the classifier head, not full backbone fine-tuning.")

Saved XLSR-head training result:
  best_epoch: 8
  dev_f1: 0.9171255506607929
  eval_f1: 0.911331529906823
  eval_recall: 0.9407384424449271
  eval_precision: 0.8837073739434567
  eval_eer: 0.4936268851474218
  confusion_matrix: [[369, 1197], [573, 9096]]
  hyperparams: {'hidden_dim': 256, 'dropout': 0.3, 'lr': 0.001, 'batch_size': 256, 'epochs_trained': 8}

Interpretation:
- best_xlsr_head.pth is the trained MLP classification head.
- XLSR-53 backbone was frozen; embeddings are stored in data/embeddings.
- This is real training of the classifier head, not full backbone fine-tuning.


### Pipeline code checks

This cell verifies that the working scripts contain the key parts of the ML pipeline: XLSR feature extraction, MLP-head training, model saving, and threshold tuning.


In [7]:
script_checks = {
    "extract_embeddings.py": ["Wav2Vec2Model", "Wav2Vec2FeatureExtractor", "np.save", "X_", "y_"],
    "train_xlsr_head.py": ["class SpoofClassifier", "DataLoader", "for epoch", "torch.save", "best_xlsr_head.pth"],
    "threshold_tuning.py": ["evaluate_threshold", "dev_probs_xlsr", "eval_probs_xlsr", "optimal_threshold"],
}

for script, markers in script_checks.items():
    path = TRAIN_DIR / script
    text = path.read_text(encoding="utf-8")
    print(script)
    for marker in markers:
        print(f"  {marker:28s}: {'OK' if marker in text else 'MISSING'}")


extract_embeddings.py
  Wav2Vec2Model               : OK
  Wav2Vec2FeatureExtractor    : OK
  np.save                     : OK
  X_                          : OK
  y_                          : OK
train_xlsr_head.py
  class SpoofClassifier       : OK
  DataLoader                  : OK
  for epoch                   : OK
  torch.save                  : OK
  best_xlsr_head.pth          : OK
threshold_tuning.py
  evaluate_threshold          : OK
  dev_probs_xlsr              : OK
  eval_probs_xlsr             : OK
  optimal_threshold           : OK


## 5. Постобработка предсказаний

Постобработка в исправленной версии должна быть скромной и проверяемой:

1. **Threshold tuning** на `dev`: выбор `t=0.37` вместо дефолтного `0.50`;
2. **Fusion layer** как дополнительный demo-слой для реальных звонков: codec,
   VoIP/IP, duration.

Главное улучшение на eval — именно threshold tuning. Fusion на 3 файлах нельзя
продавать как общую метрику модели; это демонстрация продуктового слоя.


In [8]:
def load_npy_1d(path):
    shape, dtype = npy_header(path)
    if len(shape) != 1:
        raise ValueError(f"Expected 1D npy, got {shape}")
    count = shape[0]
    with open(path, "rb") as f:
        f.read(6)
        version = tuple(f.read(2))
        if version == (1, 0):
            header_len = struct.unpack("<H", f.read(2))[0]
        else:
            header_len = struct.unpack("<I", f.read(4))[0]
        f.read(header_len)
        data = f.read()
    fmt_map = {"<f4": "f", "<f8": "d", "<i4": "i", "|u1": "B"}
    fmt = fmt_map[dtype]
    size = struct.calcsize("<" + fmt)
    return list(struct.unpack("<" + fmt * count, data[: count * size]))

def metrics(y_true, probs, threshold):
    preds = [1 if p >= threshold else 0 for p in probs]
    tp = sum(1 for y, p in zip(y_true, preds) if y == 1 and p == 1)
    tn = sum(1 for y, p in zip(y_true, preds) if y == 0 and p == 0)
    fp = sum(1 for y, p in zip(y_true, preds) if y == 0 and p == 1)
    fn = sum(1 for y, p in zip(y_true, preds) if y == 1 and p == 0)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp, "precision": precision, "recall": recall, "f1": f1}

y_eval = [int(x) for x in load_npy_1d(EMB_DIR / "y_eval.npy")]
probs_eval = [float(x) for x in load_npy_1d(MODEL_DIR / "eval_probs_xlsr.npy")]

for t in [0.50, 0.37]:
    m = metrics(y_eval, probs_eval, t)
    print(f"threshold={t:.2f}")
    print(f"  F1={m['f1']:.4f}  Recall={m['recall']:.4f}  Precision={m['precision']:.4f}")
    print(f"  TN={m['tn']} FP={m['fp']} FN={m['fn']} TP={m['tp']}")

threshold=0.50
  F1=0.9113  Recall=0.9407  Precision=0.8837
  TN=369 FP=1197 FN=573 TP=9096
threshold=0.37
  F1=0.9268  Recall=0.9805  Precision=0.8788
  TN=258 FP=1308 FN=189 TP=9480


In [9]:
with open(MODEL_DIR / "threshold_tuning_results.json", encoding="utf-8") as f:
    threshold_results = json.load(f)

print("Saved threshold tuning summary:")
print("  optimal_threshold:", threshold_results["optimal_threshold"])
print("  F1 delta:", threshold_results["improvement"]["f1_delta"])
print("  Recall delta:", threshold_results["improvement"]["recall_delta"])
print("  Cost saved:", threshold_results["improvement"]["cost_saved"])

print("\nConclusion:")
print("- Main improvement is threshold tuning: 0.50 -> 0.37.")
print("- Recall improves substantially with only small precision loss.")
print("- This directly matches the business priority: FN is more expensive than FP.")

Saved threshold tuning summary:
  optimal_threshold: 0.37000000000000005
  F1 delta: 0.015490584772748783
  Recall delta: 0.039714551659944086
  Cost saved: 7674450

Conclusion:
- Main improvement is threshold tuning: 0.50 -> 0.37.
- Recall improves substantially with only small precision loss.
- This directly matches the business priority: FN is more expensive than FP.


## 6. Анализ качества в разрезе метрик

Ниже метрики считаются не из вручную вписанной таблицы, а из сохранённых
вероятностей модели и labels. Это закрывает претензию “не видно, что оно работало”.


In [10]:
def rows_by_split(split):
    return [r for r in metadata if r["split"] == split]

eval_rows = rows_by_split("eval")
assert len(eval_rows) == len(y_eval) == len(probs_eval)

def segment_report(name, mask):
    idx = [i for i, ok in enumerate(mask) if ok]
    if not idx:
        print(f"{name:24s} n=0")
        return
    yy = [y_eval[i] for i in idx]
    pp = [probs_eval[i] for i in idx]
    m = metrics(yy, pp, 0.37)
    positives = sum(yy)
    negatives = len(yy) - positives
    if positives == 0:
        fpr = m["fp"] / (m["fp"] + m["tn"]) if (m["fp"] + m["tn"]) else 0.0
        tnr = m["tn"] / (m["fp"] + m["tn"]) if (m["fp"] + m["tn"]) else 0.0
        print(f"{name:24s} n={len(idx):5d} real-only: FPR={fpr:.4f} TNR={tnr:.4f} FP={m['fp']}")
    elif negatives == 0:
        print(f"{name:24s} n={len(idx):5d} fake-only: Recall={m['recall']:.4f} FN={m['fn']}")
    else:
        print(f"{name:24s} n={len(idx):5d} F1={m['f1']:.4f} Recall={m['recall']:.4f} Precision={m['precision']:.4f} FP={m['fp']} FN={m['fn']}")

print("Segment metrics at threshold=0.37")
segment_report("all eval", [True] * len(eval_rows))
segment_report("language=en", [r["language"] == "en" for r in eval_rows])
segment_report("language=ru", [r["language"] == "ru" for r in eval_rows])
segment_report("source=asvspoof", [r["source"] == "asvspoof2019la" for r in eval_rows])
segment_report("source=golos", [r["source"] == "golos" for r in eval_rows])
segment_report("ru_fake", [r["language"] == "ru" and r["label"] == "spoof" for r in eval_rows])

print("\nNote:")
print("RU eval is much smaller than EN eval, so RU-only estimates have higher variance.")
print("The main RU weakness is false positives on RU real (Golos/Common Voice),")
print("so the next data-centric fix should add codec-augmented RU bonafide examples.")

Segment metrics at threshold=0.37
all eval                 n=11235 F1=0.9268 Recall=0.9805 Precision=0.8788 FP=1308 FN=189
language=en              n=10686 F1=0.9358 Recall=0.9805 Precision=0.8950 FP=1102 FN=187
language=ru              n=  549 F1=0.4583 Recall=0.9778 Precision=0.2993 FP=206 FN=2
source=asvspoof          n=10686 F1=0.9358 Recall=0.9805 Precision=0.8950 FP=1102 FN=187
source=golos             n=  444 real-only: FPR=0.4369 TNR=0.5631 FP=194
ru_fake                  n=   90 fake-only: Recall=0.9778 FN=2

Note:
RU eval is much smaller than EN eval, so RU-only estimates have higher variance.
The main RU weakness is false positives on RU real (Golos/Common Voice),
so the next data-centric fix should add codec-augmented RU bonafide examples.


In [11]:
def duration_bucket(d):
    d = float(d)
    if d < 2:
        return "<2s"
    if d < 3:
        return "2-3s"
    if d < 5:
        return "3-5s"
    return ">=5s"

buckets = ["<2s", "2-3s", "3-5s", ">=5s"]
print("Duration-bucket metrics at threshold=0.37")
for b in buckets:
    segment_report(f"duration {b}", [duration_bucket(r["duration"]) == b for r in eval_rows])

print("\nThis uses the exported duration column from metadata.csv.")
print("If raw audio is submitted, durations should be remeasured from files in the notebook.")

Duration-bucket metrics at threshold=0.37
duration <2s             n=  490 F1=0.9365 Recall=0.9790 Precision=0.8974 FP=48 FN=9
duration 2-3s            n= 3483 F1=0.9313 Recall=0.9794 Precision=0.8877 FP=379 FN=63
duration 3-5s            n= 6104 F1=0.9303 Recall=0.9818 Precision=0.8840 FP=685 FN=97
duration >=5s            n= 1158 F1=0.8867 Recall=0.9769 Precision=0.8117 FP=196 FN=20

This uses the exported duration column from metadata.csv.
If raw audio is submitted, durations should be remeasured from files in the notebook.


## 7. Что именно исправлено относительно старого ДЗ7

1. Ноутбук содержит выполненные ячейки и сохранённые outputs.
2. Формулировка `fine-tune XLSR-53` уточнена: в этой работе обучался не весь backbone, а classification head.
3. Архитектура описана корректно: frozen XLSR-53 embeddings + trained MLP-head.
4. `SimplifiedAASIST` не называется AASIST; это неудачный `MelCNNBaseline`.
5. Chunking исправлен: хвост аудио не теряется.
6. Метрики `F1/Recall/Precision/CM` пересчитываются из `y_eval.npy` и `eval_probs_xlsr.npy`.
7. Fusion показан как продуктовая постобработка/demo, а не как полноценная eval-метрика.

### Финальный вывод

В этой версии работа оформлена как проверяемый ML-pipeline:

- `extract_embeddings.py` отвечает за feature extraction на frozen XLSR-53;
- `train_xlsr_head.py` обучает MLP classification head поверх сохранённых embeddings;
- `threshold_tuning.py` подбирает рабочий порог с учётом бизнес-приоритета Recall;
- notebook пересчитывает ключевые метрики из сохранённых артефактов и показывает качество по сегментам.

Полный retrain XLSR-53 backbone не заявляется: для текущего этапа используется схема `frozen XLSR-53 -> embeddings -> trained head -> threshold tuning`.
